In [7]:
import pandas as pd
import numpy as np

# Revenue

In [15]:
rev = pd.read_csv('data/raw/cbdtp_revenues_expenses.csv')

rev

,Month,Category,Amount (in Millions)
0,2025-01-01,Program Expenses,-11.14
1,2025-01-01,Toll Revenue,48.66
2,2025-02-01,Program Expenses,-11.47
3,2025-02-01,Toll Revenue,51.92
4,2025-03-01,Program Expenses,-13.28
5,2025-03-01,Toll Revenue,58.43
6,2025-04-01,Program Expenses,-10.82
7,2025-04-01,Toll Revenue,56.73
8,2025-05-01,Program Expenses,-10.94
9,2025-05-01,Toll Revenue,61.04


In [21]:
# pivot the data to wide format
rev_wide = rev.pivot(index='Month', columns='Category', values='Amount (in Millions)').reset_index()

rev_wide['Program Net Revenue'] = rev_wide['Toll Revenue'] + rev_wide['Program Expenses']

rev_wide.to_csv('data/processed/revenue.csv', index=False)

# Survey Data

In [8]:
survey_data = pd.read_csv('data/raw/survey_data.csv')

In [9]:
survey_data['percentage'] = survey_data['percentage'] / 100

boroughs_survey = survey_data[(survey_data['category'] == 'Borough') | (survey_data['category'] == 'Overall')].copy()

boroughs_survey.drop(columns=['category'], inplace=True)
boroughs_survey.rename(columns={'subcategory': 'Borough'}, inplace=True)

boroughs_survey.to_csv('data/processed/survey_data.csv', index=False)

# Traffic Fatalities

In [10]:
tf = pd.read_csv('data/raw/traffic_fatalities.csv')

In [12]:
tf.columns

Index(['Year', 'Pedestrians', 'Traditional Bike', 'E-Bike', 'Moped',
       'Stand-up Scooter', 'Motorcycle', 'Off-Road', 'Other',
       'Motor Vehicle Occupants', 'Total'],
      dtype='object')

In [13]:
tf['Motorized Two-Wheelers'] = tf['Traditional Bike'] + tf['E-Bike'] + tf['Moped'] + tf['Stand-up Scooter'] + tf['Motorcycle'] + tf['Off-Road'] + tf['Other']

tf.drop(columns=['Traditional Bike', 'E-Bike', 'Moped', 'Stand-up Scooter', 'Motorcycle', 'Off-Road', 'Other', 'Total'], inplace=True)

tf.melt(id_vars=['Year'], var_name='Vehicle Type', value_name='Fatalities').to_csv('data/processed/traffic_fatalities.csv', index=False)

# MTA Crime

In [33]:
crime = pd.read_csv('data/raw/MTA_Major_Felonies_20260120.csv')

crime['Month'] = pd.to_datetime(crime['Month'], format='%m/%d/%Y')

crime = crime[crime['Agency'] == 'NYCT']

# replace null in the Crimes per Million Riders column with 0
crime['Crimes per Million Riders'] = crime['Crimes per Million Riders'].fillna(0)

crime.to_csv('data/processed/crime_data.csv', index=False)

# Performance

In [45]:
import requests

base_url = "https://data.ny.gov/resource/"

# Define dataset IDs and query
dataset_2025 = "nmu4-7tz9.json"
dataset_2020_2024 = "bg59-42xi.json"
query = "SELECT `month`, sum(`num_sched_trains`), sum(`num_actual_trains`) GROUP BY `month`"

# Fetch data with parameters
performance_2025_response = requests.get(base_url + dataset_2025, params={"$query": query})
performance_2020_2024_response = requests.get(base_url + dataset_2020_2024, params={"$query": query + " ORDER BY `month` DESC NULL FIRST"})

performance_2025_df = pd.DataFrame(performance_2025_response.json())
performance_2020_2024_df = pd.DataFrame(performance_2020_2024_response.json())

# concatenate the two dataframes
performance = pd.concat([performance_2020_2024_df, performance_2025_df])

performance_2025_df = pd.read_json(performance_2025)
performance_2020_2024_df = pd.read_json(performance_2020_2024)

# concatenate the two dataframes
performance = pd.concat([performance_2020_2024_df, performance_2025_df])

In [46]:
performance['month'] = pd.to_datetime(performance['month'], format='%Y-%m-%dT%H:%M:%S.%f')

In [47]:
performance['service_rate'] = performance['sum_num_actual_trains'] / performance['sum_num_sched_trains']

performance

,month,sum_num_sched_trains,sum_num_actual_trains,service_rate
0,2024-12-01,78274,73881,0.943877
1,2024-11-01,75197,71620,0.952432
2,2024-10-01,74253,71221,0.959167
3,2024-09-01,73604,70542,0.958399
4,2024-08-01,75778,71977,0.949840
...,...,...,...,...
6,2025-04-01,72751,70409,0.967808
7,2025-11-01,77181,73939,0.957995
8,2025-10-01,74823,71319,0.953169
9,2025-08-01,77151,73479,0.952405


In [48]:
# restrict to 2023 onwards
performance = performance[performance['month'] >= '2023-01-01'][['month', 'service_rate']]

# Schedule Performance

In [ ]:
schedule_2020_2024_dataset = "4apg-4kt9"
schedule_2025_dataset = "s4u6-t435"

query_schedule_2020_2024 = "SELECT `month`, median(`over_five_mins_perc`), median(`customer_journey_time`), median(`additional_platform_time`), median(`additional_train_time`) WHERE caseless_one_of(`period`, \"peak\") GROUP BY `month` HAVING `month` BETWEEN \"2023-01-01T00:00:00\" :: floating_timestamp AND \"2024-12-31T23:45:00\" :: floating_timestamp ORDER BY `month` DESC NULL FIRST"

query_schedule_2025 = "SELECT `month`, median(`additional_platform_time`), median(`additional_train_time`), median(`over_five_mins_perc`), median(`customer_journey_time`) WHERE caseless_one_of(`period`, \"peak\") GROUP BY `month`"


In [67]:
schedule_2020_2024_response = requests.get(base_url + schedule_2020_2024_dataset + ".json", params={"$query": query_schedule_2020_2024})
schedule_2025_response = requests.get(base_url + schedule_2025_dataset + ".json", params={"$query": query_schedule_2025})

schedule_2020_2024_df = pd.DataFrame(schedule_2020_2024_response.json())
schedule_2025_df = pd.DataFrame(schedule_2025_response.json())

schedule = pd.concat([schedule_2020_2024_df, schedule_2025_df])

schedule

,message,errorCode,data,month,median_additional_platform_time,median_additional_train_time,median_over_five_mins_perc,median_customer_journey_time
column,Query coordinator error: query.soql.no-such-co...,query.soql.no-such-column,customer_journey_time_performance,NaN,NaN,NaN,NaN,NaN
dataset,Query coordinator error: query.soql.no-such-co...,query.soql.no-such-column,india.16324,NaN,NaN,NaN,NaN,NaN
position,Query coordinator error: query.soql.no-such-co...,query.soql.no-such-column,"{'row': 1, 'column': 227, 'line': 'SELECT `mon...",NaN,NaN,NaN,NaN,NaN
0,NaN,NaN,NaN,2025-01-01T00:00:00.000,1.1808805203851,0.541543733162861,0.140231843660787,0.859768156339213
1,NaN,NaN,NaN,2025-02-01T00:00:00.000,1.18813293753114,0.3952486253134,0.128381308520911,0.871618691479089
2,NaN,NaN,NaN,2025-03-01T00:00:00.000,1.08952874442692,0.451986887926494,0.124321694066426,0.875678305933574
3,NaN,NaN,NaN,2025-04-01T00:00:00.000,1.17124969822762,0.362731170220644,0.118470243651637,0.881529756348363
4,NaN,NaN,NaN,2025-05-01T00:00:00.000,1.08783895806876,0.351808015852665,0.113657079091963,0.886342920908037
5,NaN,NaN,NaN,2025-06-01T00:00:00.000,1.04481317373737,0.29776221707862,0.1237674537622,0.8762325462378
6,NaN,NaN,NaN,2025-07-01T00:00:00.000,1.38902580062358,0.43812211596766,0.143363683642495,0.856636316357505


In [68]:
schedule_2025_query_messy = "https://data.ny.gov/resource/s4u6-t435.json?$query=SELECT%0A%20%20%60month%60%2C%0A%20%20median(%60additional_platform_time%60)%2C%0A%20%20median(%60customer_journey_time%60)%2C%0A%20%20median(%60additional_train_time%60)%2C%0A%20%20median(%60over_five_mins_perc%60)%0AWHERE%20caseless_one_of(%60period%60%2C%20%22peak%22)%0AGROUP%20BY%20%60month%60"